In [0]:
/* In databricks to load a table we use catalog.
In catalog we do the following to load the table:
1. Go to catalog present on the left hand side in datbricks
2. In catalog click on the '+' sign present on the left hand side and then click on add data
3. Now click on upload files to a volume
4. Browse the file/ folder that you want to upload from your system
5. Now go down and click on create volume.
6. In create volume keep:
a) volume name= my_volume
b) Catalog = workspace
c) Schema = default
7. Now click on upload, and wait for it to get uploaded*/
USE CATALOG workspace;
USE SCHEMA default;
/*The name of the table should be written as 'catalog.schema.table', to avoid such a big name on the top of the notebook we write these 2 steps so that we can reduce the table name to just table, rather than writing 'catalog.schema.table'*/

In [0]:
/* Before creating the table drop function is used to check if any table with the same name exists then the table will be dropped so that there is no issue in creating the table*/
DROP TABLE IF EXISTS workspace.default.superstore;
CREATE TABLE workspace.default.superstore AS
SELECT * FROM read_files(
  '/Volumes/workspace/default/my_volume/Superstore.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

num_affected_rows,num_inserted_rows


In [0]:
-- Verifying if the dataset was loaded successfully
SELECT * FROM superstore LIMIT 10;

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit,_rescued_data
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0,41.9136,null
2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.94,3,0,219.582,null
3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62,2,0,6.8714,null
4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.031,null
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164,null
6,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,FUR-FU-10001487,Furniture,Furnishings,"Eldon Expressions Wood and Plastic Desk Accessories, Cherry Wood",48.86,7,0,14.1694,null
7,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.28,4,0,1.9656,null
8,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.152,6,0.2,90.7152,null
9,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-BI-10003910,Office Supplies,Binders,DXL Angle-View Binders with Locking Rings by Samsill,18.504,3,0.2,5.7825,null
10,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AP-10002892,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.9,5,0,34.47,null


In [0]:
-- This command tells us about the number of rows the loaded table contains
SELECT COUNT(*) FROM superstore;
-- The table contains 9994 rows

COUNT(*)
9994


In [0]:
-- 1) Schema of the table
DESCRIBE TABLE superstore;
/*Describe table tells about the col_names in the table, as well as their datatypes along with their comments
In some other sql platforms 'DESCRIBE TABLE' also specifies the constraints applied on the particular columns such as primary and foreign key, but as in databricks both the keys are not enforced and are only used for informational use , hence they are not mentioned here. */

col_name,data_type,comment
Row_ID,int,null
Order_ID,string,null
Order_Date,date,null
Ship_Date,date,null
Ship_Mode,string,null
Customer_ID,string,null
Customer_Name,string,null
Segment,string,null
Country,string,null
City,string,null


In [0]:
/*From describe table we can see that Sales, discount and quantity has datatype as string but these are numeric values so their datatypes should be changed to either int, float or double
Databricks supports type widening using ALTER command only if the type is same that is it can change int to double but not string to double.
Hence we should re-create the table again so that we can manually specify the required datatypes*/
CREATE OR REPLACE TABLE workspace.default.superstore AS
SELECT
  Row_ID,
  Order_ID,
  Order_Date,
  Ship_Date,
  Ship_Mode,
  Customer_ID,
  Customer_Name,
  Segment,
  Country,
  City,
  State,
  TRY_CAST(Postal_Code AS INT) AS Postal_Code,
  Region,
  Product_ID,
  Category,
  Sub_Category,
  Product_Name,
  TRY_CAST(Sales AS DOUBLE) AS Sales,
  TRY_CAST(Quantity AS INT) AS Quantity,
  TRY_CAST(Discount AS DOUBLE) AS Discount,
  TRY_CAST(Profit AS DOUBLE) AS Profit
FROM read_files(
  '/Volumes/workspace/default/my_volume/Superstore.csv',
  format => 'csv',
  header => true,
  inferSchema => false
);
/* Here TRY_CAST is used so that even if there was an error in reading/parsing the csv file or if the columns that i wanted to convert to a different datatype in that some of the rows did not support the conversion then TRY_CAST cahnges those values to null instead of giving an error , whereas if I had used CAST it would have given an error if it found any incompatible datatype during the datatype change.*/


num_affected_rows,num_inserted_rows


In [0]:
-- Veryfying if the datatypes were changed successfully
DESCRIBE TABLE superstore;
-- Now the datatypes have been changed successfully

col_name,data_type,comment
Row_ID,string,null
Order_ID,string,null
Order_Date,string,null
Ship_Date,string,null
Ship_Mode,string,null
Customer_ID,string,null
Customer_Name,string,null
Segment,string,null
Country,string,null
City,string,null


In [0]:
/*2) Sample data
To show sample data, we use select statement, but as the data is very huge we will limit the number of rows.*/
SELECT * FROM superstore LIMIT 10;
-- The number of rows has been limited to 10 i.e. only 10 rows are displayed   

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.94,3,0.0,219.582
3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62,2,0.0,6.8714
4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.031
5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164
6,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,FUR-FU-10001487,Furniture,Furnishings,"Eldon Expressions Wood and Plastic Desk Accessories, Cherry Wood",48.86,7,0.0,14.1694
7,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.28,4,0.0,1.9656
8,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.152,6,0.2,90.7152
9,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-BI-10003910,Office Supplies,Binders,DXL Angle-View Binders with Locking Rings by Samsill,18.504,3,0.2,5.7825
10,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AP-10002892,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.9,5,0.0,34.47


In [0]:
SELECT * FROM superstore LIMIT 20;
/* Limit is used to reduce the number of rows that is it does not show the whole table but only shows the rows that the user wants */

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.94,3,0.0,219.582
3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62,2,0.0,6.8714
4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.031
5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164
6,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,FUR-FU-10001487,Furniture,Furnishings,"Eldon Expressions Wood and Plastic Desk Accessories, Cherry Wood",48.86,7,0.0,14.1694
7,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.28,4,0.0,1.9656
8,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.152,6,0.2,90.7152
9,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-BI-10003910,Office Supplies,Binders,DXL Angle-View Binders with Locking Rings by Samsill,18.504,3,0.2,5.7825
10,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AP-10002892,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.9,5,0.0,34.47


In [0]:
-- 1) Region
SELECT DISTINCT Region FROM superstore;
/*DISTINCT is used to retrieve the unique rows in a given table / column
Here in region we have 4 different regions that are:
a) South
b) West
c) Central
d) East*/

Region
South
West
Central
East


In [0]:
-- Filtering the table based on region = south 
SELECT * FROM superstore WHERE Region = 'South' LIMIT 10;

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.94,3,0.0,219.582
4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.031
5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164
13,CA-2017-114412,4/15/2017,4/20/2017,Standard Class,AA-10480,Andrew Allen,Consumer,United States,Concord,North Carolina,28027,South,OFF-PA-10002365,Office Supplies,Paper,Xerox 1967,15.552,3,0.2,5.4432
44,CA-2017-139619,9/19/2017,9/23/2017,Standard Class,ES-14080,Erin Smith,Corporate,United States,Melbourne,Florida,32935,South,OFF-ST-10003282,Office Supplies,Storage,"Advantus 10-Drawer Portable Organizer, Chrome Metal Frame, Smoke Drawers",95.616,2,0.2,9.5616
70,CA-2016-119823,6/4/2016,6/6/2016,First Class,KD-16270,Karen Daniels,Consumer,United States,Springfield,Virginia,22153,South,OFF-PA-10000482,Office Supplies,Paper,"Snap-A-Way Black Print Carbonless Ruled Speed Letter, Triplicate",75.88,2,0.0,35.6636
73,US-2015-134026,4/26/2015,5/2/2015,Standard Class,JE-15745,Joel Eaton,Consumer,United States,Memphis,Tennessee,38109,South,FUR-CH-10000513,Furniture,Chairs,High-Back Leather Manager's Chair,831.936,8,0.2,-114.3912
74,US-2015-134026,4/26/2015,5/2/2015,Standard Class,JE-15745,Joel Eaton,Consumer,United States,Memphis,Tennessee,38109,South,FUR-FU-10003708,Furniture,Furnishings,"""Tenex Traditional Chairmats for Medium Pile Carpet, Standard Lip, 36"""" x 48""""""",97.04,2,0.2,1.213
75,US-2015-134026,4/26/2015,5/2/2015,Standard Class,JE-15745,Joel Eaton,Consumer,United States,Memphis,Tennessee,38109,South,OFF-ST-10004123,Office Supplies,Storage,Safco Industrial Wire Shelving System,72.784,1,0.2,-18.196


In [0]:
-- Finding number of rows with region=south
SELECT COUNT(*) FROM superstore WHERE Region = 'South';
-- The table contains 1620 records where their region is mentioned as south

COUNT(*)
1620


In [0]:
-- Finding the number of records region wise using group by
SELECT Region, COUNT(*) AS No_of_records_per_region FROM superstore GROUP BY Region;

Region,No_of_records_per_region
South,1620
West,3203
Central,2323
East,2848


In [0]:
-- Using LIKE operator in WHERE Clause
SELECT Row_ID,Customer_Name,Region FROM superstore WHERE Region like '%st' LIMIT 20;
-- It displays all the rows where the region ends with 'st', hence it shows all the rows containing region 'east' and 'west'

Row_ID,Customer_Name,Region
3,Darrin Van Huff,West
6,Brosina Hoffman,West
7,Brosina Hoffman,West
8,Brosina Hoffman,West
9,Brosina Hoffman,West
10,Brosina Hoffman,West
11,Brosina Hoffman,West
12,Brosina Hoffman,West
14,Irene Maddox,West
18,Alejandro Grove,West


In [0]:
-- 2) Category
SELECT DISTINCT Category FROM superstore;
/*DISTINCT is used to retrieve the unique rows in a given table / column
Here in category we have 3 different categories that are:
a) Office Supplies
b) Technology
c) Furniture*/

Category
Furniture
Office Supplies
Technology


In [0]:
/* Filtering the table by category, where category=furniture and the sales are greater than 1000, which will be displayed in descending order */
SELECT Row_ID,Order_ID,Customer_Name,Category,Sales FROM superstore WHERE Category='Furniture' AND Sales > 1000 ORDER BY sales desc LIMIT 20;

Row_ID,Order_ID,Customer_Name,Category,Sales
7244,CA-2017-118892,Tom Prescott,Furniture,4416.174
9742,CA-2015-117086,Quincy Jones,Furniture,4404.9
9640,CA-2015-116638,Joseph Holt,Furniture,4297.644
5918,US-2015-126977,Peter Fuller,Furniture,4228.704
6536,CA-2014-128209,Greg Tran,Furniture,4007.84
9426,CA-2014-116246,Luke Weiss,Furniture,3785.292
1247,CA-2014-168494,Nora Preis,Furniture,3610.848
5171,CA-2016-122903,Laura Armstrong,Furniture,3504.9
9650,CA-2016-107104,Maribeth Schnelling,Furniture,3406.664
9858,CA-2015-164301,Ellis Ballard,Furniture,3393.68


In [0]:
-- Finding the total sales done in the Category Technology
SELECT SUM(Sales) AS Total_Sales_in_Technology FROM superstore where Category='Technology'; 

Total_Sales_in_Technology
835900.0669999964


In [0]:
-- 3) Date
-- Using between operator to find the customers who did orders between 2015 and 2016
SELECT Row_ID, Order_ID, Order_Date, Customer_Name FROM superstore WHERE Order_Date BETWEEN '01-01-2015' AND '31-12-2016' LIMIT 20;
-- It shows 5075 records meaning that 5075 orders were made in the years 2015 and 2016

Row_ID,Order_ID,Order_Date,Customer_Name
1,CA-2016-152156,11/8/2016,Claire Gute
2,CA-2016-152156,11/8/2016,Claire Gute
4,US-2015-108966,10/11/2015,Sean O'Donnell
5,US-2015-108966,10/11/2015,Sean O'Donnell
14,CA-2016-161389,12/5/2016,Irene Maddox
15,US-2015-118983,11/22/2015,Harold Pawlan
16,US-2015-118983,11/22/2015,Harold Pawlan
17,CA-2014-105893,11/11/2014,Pete Kriz
22,CA-2016-137330,12/9/2016,Ken Black
23,CA-2016-137330,12/9/2016,Ken Black


In [0]:
-- Counting the number of orders per year
SELECT YEAR(TO_DATE(Order_Date, 'M/d/yyyy')) AS Year, COUNT(*) AS No_of_order_per_year FROM superstore GROUP BY YEAR(TO_DATE(Order_Date, 'M/d/yyyy')) ORDER BY Year;
-- In the table the date is in the form of dd-mm-yyyy, but databricks only recognizes the format mm-dd-yyyy, hence TO_DATE was applied, if it was not applied, it would have given an error saying that date operation cannot be performed as it databricks would have read it as a string and not as date.

Year,No_of_order_per_year
2014,1993
2015,2102
2016,2587
2017,3312


In [0]:
-- 4) Sales
SELECT Order_Date,Customer_Name,State,City,Region,Sales FROM superstore WHERE Sales BETWEEN 2000 AND 3000 LIMIT 10;
-- There are 77 records that have Sales Between 1000 and 2000(Both inclusive)

Order_Date,Customer_Name,State,City,Region,Sales
6/1/2014,Dianna Wilson,Minnesota,Lakeville,Central,2001.86
9/19/2014,Sean Braxton,Texas,Houston,Central,2519.958
9/8/2016,Anna Gayman,Texas,Houston,Central,2396.2656
10/29/2014,Sue Ann Reed,Illinois,Chicago,Central,2735.952
1/22/2017,Alan Dominguez,Montana,Great Falls,West,2999.95
11/28/2017,Ross Baird,Pennsylvania,Philadelphia,East,2065.32
5/21/2014,Mitch Willingham,Virginia,Virginia Beach,South,2715.93
7/31/2015,Rick Hansen,New York,New York City,East,2309.65
12/20/2014,Penelope Sewall,Virginia,Harrisonburg,South,2244.48
10/15/2015,Joel Eaton,Texas,Amarillo,Central,2453.43


In [0]:
SELECT Order_ID, Customer_Name,Region ,Sales FROM superstore WHERE Sales>3000 AND Region like 'C%';
-- Here in the dataset there are only 12 rows in the region 'Central', where sales is greater than 3000

Order_ID,Customer_Name,Region,Sales
CA-2014-139892,Becky Martin,Central,8159.952
US-2014-106992,Sean Braxton,Central,3059.982
CA-2017-159366,Bart Watters,Central,3059.982
CA-2014-116904,Sanjit Chand,Central,9449.95
CA-2016-122903,Laura Armstrong,Central,3504.9
CA-2016-103982,Alex Avila,Central,3930.072
CA-2017-138289,Andy Reiter,Central,5443.96
US-2017-167402,Cathy Prescott,Central,4164.05
CA-2016-118689,Tamara Chand,Central,17499.95
CA-2015-120782,Shirley Daniels,Central,3812.97


In [0]:
-- 1) Sales
/* With GROUP BY we can also use aggregate functions. These functions are:
1) Sum() -> Sums all the records in the specified column
2) Max() -> Selects the maximum value in a column
3) Min() -> Selects the minimum value in a column
4) Count() -> counts the number of records per column
5) Avg() -> Finds the abg value per column*/
-- Finding total amount of sales done per region
SELECT Region, SUM(Sales) AS Total_Sales_Per_Region FROM superstore GROUP BY Region;

Region,Total_Sales_Per_Region
South,388983.58500000037
West,713471.3445000004
Central,497800.8728000007
East,672194.0539999981


In [0]:
-- Finding maximum Sales per Region
SELECT Region, MAX(Sales) AS Max_Sale_Per_Region FROM superstore GROUP BY Region ORDER BY Max(Sales) desc;
-- Therefore the highest Sales was done in the Southern Region i.e

Region,Max_Sale_Per_Region
South,22638.48
Central,17499.95
West,13999.96
East,11199.968


In [0]:
-- Finding the Minimum Sales done per State and City
SELECT State, City, MIN(Sales) FROM superstore WHERE Sales IS NOT NULL GROUP BY State,City ORDER BY MIN(Sales) ASC LIMIT 20;

State,City,MIN(Sales)
Texas,Houston,0.444
Texas,Waco,0.556
Illinois,Chicago,0.836
Pennsylvania,Philadelphia,0.852
Texas,Mesquite,0.876
Texas,Huntsville,0.898
California,San Francisco,0.99
Texas,Dallas,1.044
Oregon,Portland,1.08
Colorado,Aurora,1.08


In [0]:
-- 2) Category
/* It displays the category wise total sales(sum of sales per category) , total quantity (total quantity per category), Average profit per category as well as the total number of orders placed per category, all decimal values rounded upto 2 decimal places.*/ 
SELECT Category,ROUND(SUM(Sales), 2) AS Total_Sales,SUM(Quantity) AS Total_Quantity,ROUND(AVG(Profit), 2) AS Avg_Profit,COUNT(*) AS Total_Orders FROM superstore GROUP BY Category ORDER BY Total_Sales DESC;

Category,Total_Sales,Total_Quantity,Avg_Profit,Total_Orders
Technology,835900.07,6904,78.72,1847
Furniture,733046.86,7855,9.28,2121
Office Supplies,703502.93,21990,20.02,6026


In [0]:
-- 3) By Region
SELECT Region,ROUND(SUM(Sales), 2) AS Total_Sales,ROUND(SUM(Profit), 2) AS Total_Profit,ROUND(AVG(Sales), 2) AS Avg_Order_Value FROM superstore GROUP BY Region ORDER BY Total_Sales DESC;

Region,Total_Sales,Total_Profit,Avg_Order_Value
West,713471.34,107303.7,230.23
East,672194.05,91603.06,243.9
Central,497800.87,40150.5,220.27
South,388983.59,46650.34,246.35


In [0]:
-- 4) By Sub-Category
/* It displays the category and sub-category wise total sales(sum of sales per sub category in every category) as well as total profits per sub category in category*/
SELECT Category,Sub_Category,ROUND(SUM(Sales), 2) AS Total_Sales,ROUND(SUM(Profit), 2) AS Total_Profit FROM superstore GROUP BY Category,Sub_Category ORDER BY Total_Sales DESC;

Category,Sub_Category,Total_Sales,Total_Profit
Technology,Phones,329753.09,44449.08
Furniture,Chairs,328449.1,26590.17
Office Supplies,Storage,216803.21,21529.91
Furniture,Tables,206965.53,-17725.48
Office Supplies,Binders,199905.72,30038.82
Technology,Machines,189238.63,3384.76
Technology,Accessories,167380.32,41936.64
Technology,Copiers,149528.03,55617.82
Furniture,Bookcases,114880.0,-3472.56
Office Supplies,Appliances,107532.16,18138.01


In [0]:
/* 1) Top Products and Top Category i.e the names of the products and the category and sub-category they belong to depending upon their sales
LIMIT 10 will only display the top 10 rows */
SELECT Category, Sub_Category, Product_Name, ROUND(AVG(Sales),2) AS Avg_Sales FROM superstore GROUP BY Category, Sub_Category, Product_Name ORDER BY AVG(Sales) DESC LIMIT 10;

Category,Sub_Category,Product_Name,Avg_Sales
Technology,Machines,Cisco TelePresence System EX90 Videoconferencing Unit,22638.48
Technology,Copiers,Canon imageCLASS 2200 Advanced Copier,12319.96
Technology,Machines,Cubify CubeX 3D Printer Triple Head Print,7999.98
Technology,Machines,"3D Systems Cube Printer, 2nd Generation, Magenta",7149.95
Technology,Machines,"""HP Designjet T520 Inkjet Large Format Printer - 24"""" Color""",6124.97
Office Supplies,Supplies,High Speed Automatic Electric Letter Opener,5676.77
Office Supplies,Binders,Ibico EPK-21 Electric Binding System,5291.97
Technology,Machines,Lexmark MX611dhe Monochrome Laser Printer,4207.48
Technology,Machines,Canon imageCLASS MF7460 Monochrome Digital Laser Multifunction Copier,3991.98
Technology,Machines,Okidata MB760 Printer,3917.2


In [0]:
/* 2) Top Products and Top Category i.e the names of the products and the category and sub-category they belong to depending upon their quantity of products sold
LIMIT 10 will only display the top 10 rows */
SELECT Category, Sub_Category, Product_Name, SUM(Quantity) AS Total_Quantity FROM superstore GROUP BY Category, Sub_Category, Product_Name ORDER BY SUM(Quantity) DESC LIMIT 10;

Category,Sub_Category,Product_Name,Total_Quantity
Office Supplies,Fasteners,Staples,215
Office Supplies,Envelopes,Staple envelope,170
Office Supplies,Paper,Easy-staple paper,150
Office Supplies,Art,Staples in misc. colors,86
Furniture,Tables,KI Adjustable-Height Table,74
Office Supplies,Binders,Avery Non-Stick Binders,71
Office Supplies,Binders,Storex Dura Pro Binders,71
Office Supplies,Binders,GBC Premium Transparent Covers with Diagonal Lined Pattern,67
Furniture,Chairs,"Situations Contoured Folding Chairs, 4/Set",64
Furniture,Furnishings,Staple-based wall hangings,62


In [0]:
/* 3) Top Products and Top Category i.e the names of the products and the category and sub-category they belong to depending upon the profits made
LIMIT 10 will only display the top 10 rows */
SELECT Category, Sub_Category, Product_Name, ROUND(AVG(Profit),2) AS Avg_Profits FROM superstore GROUP BY Category, Sub_Category, Product_Name ORDER BY AVG(Profit) DESC LIMIT 10;

Category,Sub_Category,Product_Name,Avg_Profits
Technology,Copiers,Canon imageCLASS 2200 Advanced Copier,5039.99
Technology,Machines,Canon imageCLASS MF7460 Monochrome Digital Laser Multifunction Copier,1995.99
Technology,Machines,Ativa V4110MDD Micro-Cut Shredder,1886.47
Technology,Machines,"3D Systems Cube Printer, 2nd Generation, Magenta",1858.99
Technology,Machines,Zebra ZM400 Thermal Label Printer,1671.77
Technology,Machines,Hewlett-Packard Desktjet 6988DT Refurbished Printer,1668.21
Technology,Machines,Hewlett-Packard Deskjet 3050a All-in-One Color Inkjet Printer,1459.2
Technology,Machines,"""HP Designjet T520 Inkjet Large Format Printer - 24"""" Color""",1364.99
Technology,Copiers,Canon PC1060 Personal Laser Copier,1142.73
Office Supplies,Binders,Ibico EPK-21 Electric Binding System,1115.09


In [0]:
-- 1) Monthly Trends
-- It shows the monthly trends of sales and profits i.e. it displays sales as well as profits per month every year
SELECT YEAR(TO_DATE(Order_Date, 'M/d/yyyy')) AS Year,MONTH(TO_DATE(Order_Date, 'M/d/yyyy')) AS Month,ROUND(SUM(Sales), 2) AS Monthly_Sales,ROUND(SUM(Profit), 2) AS Monthly_Profit FROM superstore GROUP BY YEAR(TO_DATE(Order_Date, 'M/d/yyyy')), MONTH(TO_DATE(Order_Date, 'M/d/yyyy')) ORDER BY Year, Month ASC LIMIT 20;

Year,Month,Monthly_Sales,Monthly_Profit
2014,1,14161.35,2417.09
2014,2,4119.82,786.04
2014,3,55526.2,419.43
2014,4,28139.56,3453.56
2014,5,23634.67,2732.58
2014,6,34509.0,5058.24
2014,7,33500.87,-925.1
2014,8,27603.51,5354.0
2014,9,81496.81,8215.78
2014,10,31394.94,3453.92


In [0]:
-- It shows the monthly trends of Total Quantity of products sold every month per category in ascending order
SELECT YEAR(TO_DATE(Order_Date, 'M/d/yyyy')) AS Year,MONTH(TO_DATE(Order_Date, 'M/d/yyyy')) AS Month, Category, SUM(Quantity) AS Total_Quantity FROM superstore GROUP BY YEAR(TO_DATE(Order_Date, 'M/d/yyyy')), MONTH(TO_DATE(Order_Date, 'M/d/yyyy')), Category ORDER BY Year, Month, Total_Quantity ASC LIMIT 20;

Year,Month,Category,Total_Quantity
2014,1,Technology,45
2014,1,Furniture,70
2014,1,Office Supplies,156
2014,2,Furniture,20
2014,2,Technology,33
2014,2,Office Supplies,95
2014,3,Technology,84
2014,3,Furniture,131
2014,3,Office Supplies,362
2014,4,Furniture,81


In [0]:
-- 2) Top Customers
/* Top 10 Customers by Revenue i.e it displays the top 10 customer names that generated the highest sales/ revenue for the store */
SELECT Customer_Name,ROUND(SUM(Sales), 2) AS Total_Revenue,COUNT(DISTINCT Order_ID) AS Total_Orders FROM superstore GROUP BY Customer_Name ORDER BY Total_Revenue DESC LIMIT 10;

Customer_Name,Total_Revenue,Total_Orders
Sean Miller,25043.05,5
Tamara Chand,19017.85,5
Raymond Buch,15117.34,6
Tom Ashbrook,14595.62,4
Adrian Barton,14355.61,10
Sanjit Chand,14142.33,9
Ken Lonsdale,14071.92,12
Hunter Lopez,12873.3,6
Sanjit Engle,12209.44,11
Christopher Conant,12129.07,5


In [0]:
-- 3) Duplicates
-- Checking for duplicates
SELECT Order_ID, Product_Name, COUNT(*) AS Count FROM superstore GROUP BY Order_ID, Product_Name HAVING COUNT(*) > 1;
/* It displays the duplicate Order_ID and Product_Name i.e these peoducts were ordered again*/

Order_ID,Product_Name,Count
CA-2016-129714,Xerox 1881,2
US-2016-123750,Imation�Secure+ Hardware Encrypted USB 2.0�Flash Drive; 16GB,2
CA-2016-137043,"Electrix Architect's Clamp-On Swing Arm Lamp, Black",2
CA-2017-152912,Adjustable Depth Letter/Legal Cart,2
US-2014-150119,"Global Leather Highback Executive Chair with Pneumatic Height Adjustment, Black",2
CA-2015-103135,"GBC Prepunched Paper, 19-Hole, for Binding Systems, 24-lb",2
CA-2017-118017,Memorex Micro Travel Drive 16 GB,2
CA-2016-140571,Xerox 1964,2


In [0]:
-- 4) Profit Classification using CASE
/* Dividing profit into 4 classes:
1) Loss -> Profit<0
2) Low Profit -> 0<=Profit<=100
3) Moderate Profit -> 100<Profit<=500
4) High Profit -> Profit>500
*/
SELECT Customer_Name, Sales, Profit,
  CASE
    WHEN Profit < 0 THEN 'Loss'
    WHEN Profit BETWEEN 0 AND 100 THEN 'Low Profit'
    WHEN Profit BETWEEN 100 AND 500 THEN 'Moderate Profit'
    ELSE 'High Profit'
  END AS Profit_Category
FROM superstore LIMIT 20;

Customer_Name,Sales,Profit,Profit_Category
Claire Gute,261.96,41.9136,Low Profit
Claire Gute,731.94,219.582,Moderate Profit
Darrin Van Huff,14.62,6.8714,Low Profit
Sean O'Donnell,957.5775,-383.031,Loss
Sean O'Donnell,22.368,2.5164,Low Profit
Brosina Hoffman,48.86,14.1694,Low Profit
Brosina Hoffman,7.28,1.9656,Low Profit
Brosina Hoffman,907.152,90.7152,Low Profit
Brosina Hoffman,18.504,5.7825,Low Profit
Brosina Hoffman,114.9,34.47,Low Profit


In [0]:
-- 1) Row Count
SELECT COUNT(*) AS Total_Rows FROM superstore;
-- It displays that the table contains 9994 rows in total

Total_Rows
9994


In [0]:
-- 2) Distinct Counts
SELECT COUNT(DISTINCT Customer_Name) AS Distinct_Customers,COUNT(DISTINCT Order_ID) AS Distinct_Orders, COUNT(DISTINCT Product_ID) AS Distinct_Products FROM superstore;
-- It displays that there are 793 distinct customers, 5009 distinct orders and 1862 distinct products in the table


Distinct_Customers,Distinct_Orders,Distinct_Products
793,5009,1862


In [0]:
-- 3) Data Quality
-- 1. Checking for NULL Values
-- Null check
SELECT COUNT(*) - COUNT(Order_ID) AS Null_Order_ID,COUNT(*) - COUNT(Sales) AS Null_Sales,COUNT(*) - COUNT(Profit) AS Null_Profit,COUNT(*) - COUNT(Region) AS Null_Region FROM superstore;
/* Here we do count(*)-count(col_name), count(*) counts null values as well, but count(col_name) counts only the non-null values.
Hence their difference gives the total number of null values per mentioned column.
Here only sales have null values i.e. it contains a total of 300 null records
*/

Null_Order_ID,Null_Sales,Null_Profit,Null_Region
0,300,0,0


In [0]:
-- Finding the range of sales and profits i.e. its min, max and avg values
SELECT MIN(Sales) AS Min_Sale,MAX(Sales) AS Max_Sale,ROUND(AVG(Sales), 2) AS Avg_Sale,ROUND(MIN(Profit),2) AS Min_Profit,ROUND(MAX(Profit),2) AS Max_Profit, ROUND(AVG(Profit),2) AS Avg_Profit FROM superstore;

Min_Sale,Max_Sale,Avg_Sale,Min_Profit,Max_Profit,Avg_Profit
0.444,22638.48,234.42,-6599.98,8399.98,28.59
